# Quill — Notebook 03: Hadamard & Rotation Gates

In Notebook 02, an **X gate** flipped one selected bit in a Knot. Now we’ll make that bit quantum-mechanically uncertain before measurement.

**Goal:** prepare a Knot that can be measured as either `H` or `I`, then change the probabilities with a rotation gate.

We’ll use one 8-qubit Knot for one ASCII character. Qubit `0` is the least-significant bit (LSB), even though binary strings are usually printed most-significant bit first.

## 1. The key idea: superposition

A qubit can be in a basis state, $|0\rangle$ or $|1\rangle$, or in a superposition:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

The complex amplitudes determine measurement probabilities:

$$P(0)=|\alpha|^2, \qquad P(1)=|\beta|^2$$

For a valid state, $|\alpha|^2+|\beta|^2=1$. A measurement returns one result, not both. Repeating the same experiment many times lets us estimate the probabilities.

## 2. Hadamard gate: equal probabilities

The Hadamard gate is

$$H = \frac{1}{\sqrt{2}}\begin{bmatrix}1&1\\1&-1\end{bmatrix}$$

Applied to $|0\rangle$, it gives

$$H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}}$$

Both outcomes have probability $1/2$. Applied to $|1\rangle$, it gives $(|0\rangle-|1\rangle)/\sqrt{2}$; the minus sign is a relative phase, not a negative probability.

## 3. Why `H` and `I` make a good example

ASCII `H` is decimal 72 (`01001000`); ASCII `I` is decimal 73 (`01001001`). They differ only in the least-significant bit.

So if we prepare `H` and apply a Hadamard to qubit 0, the other seven bits stay fixed while the last bit becomes a superposition:

$$|H\rangle_K \xrightarrow{H_0} \frac{|H\rangle_K+|I\rangle_K}{\sqrt{2}}$$

Here $|H\rangle_K$ and $|I\rangle_K$ mean the 8-qubit basis states that encode those ASCII characters. The gate does not understand letters; it acts on the encoded bit.

## 4. Set up Qiskit

Run this cell in Google Colab. If the imports fail, install the packages first.

In [ ]:
%pip -q install qiskit qiskit-aer

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

simulator = AerSimulator()

## 5. Prepare `H`, then apply Hadamard to its LSB

We prepare the ASCII basis state using X gates wherever the binary encoding has a `1`. Since qubit 0 is the LSB, it is the rightmost bit of the displayed binary string.

In [ ]:
def prepare_ascii(circuit, char, offset=0):
    value = ord(char)
    for bit in range(8):  # bit 0 is the least-significant bit
        if (value >> bit) & 1:
            circuit.x(offset + bit)

qc = QuantumCircuit(8, 8)
prepare_ascii(qc, "H")
qc.h(0)
qc.measure(range(8), range(8))

display(qc.draw("mpl"))

### What do we expect?

Before measurement, the ideal state is

$$|\psi\rangle=\frac{|H\rangle_K+|I\rangle_K}{\sqrt{2}}$$

Therefore, $P(H)=1/2$ and $P(I)=1/2$. One run gives only one character. A batch of runs should show roughly equal counts, with ordinary statistical variation.

In [ ]:
counts = simulator.run(transpile(qc, simulator), shots=1000).result().get_counts()
plot_histogram(counts)

In [ ]:
def decode_byte(bitstring):
    # Qiskit count keys are displayed classical bits in reverse register order.
    return chr(int(bitstring, 2))

print({decode_byte(bits): count for bits, count in counts.items()})

**Read the result:** you should see `H` and `I` with counts near 500 each. The exact counts will vary between runs. This is a probability distribution, not a promise of exactly 50/50 in every finite sample.

## 6. Rotation gate: tune the probabilities

A rotation about the Y axis is

$$R_y(\theta)=\begin{bmatrix}\cos(\theta/2)&-\sin(\theta/2)\\\sin(\theta/2)&\cos(\theta/2)\end{bmatrix}$$

On $|0\rangle$ it produces

$$R_y(\theta)|0\rangle=\cos(\theta/2)|0\rangle+\sin(\theta/2)|1\rangle$$

So the probabilities are $P(0)=\cos^2(\theta/2)$ and $P(1)=\sin^2(\theta/2)$. At $\theta=\pi/2$, they are equal. At $\theta=\pi/3$, they are $3/4$ and $1/4$.

For our `H` Knot, qubit 0 starts at $|0\rangle$. Rotating it changes the chance of measuring `H` versus `I`.

In [ ]:
import math

theta = math.pi / 3  # 60 degrees: predicted probabilities 75% / 25%
rot = QuantumCircuit(8, 8)
prepare_ascii(rot, "H")
rot.ry(theta, 0)
rot.measure(range(8), range(8))

display(rot.draw("mpl"))
rot_counts = simulator.run(transpile(rot, simulator), shots=2000).result().get_counts()
plot_histogram(rot_counts)
print({decode_byte(bits): count for bits, count in rot_counts.items()})

You should see approximately 75% `H` and 25% `I`. The exact counts vary. This is a rotation of the **qubit state**, not a semantic transformation of a letter.

## 7. Try it yourself

1. Change `theta` to `math.pi / 2`. What probabilities do you predict?
2. Change it to `math.pi`. Which character should you measure almost every time?
3. Try a negative angle such as `-math.pi / 3`. What changes about the measurement probabilities?

**Important:** probabilities depend on the squared magnitudes of amplitudes. A negative amplitude does not mean a negative probability; its phase can matter later when states interfere.

## Takeaway

- **Hadamard:** creates equal-amplitude superposition from a basis state.
- **Rotation:** lets us tune amplitudes and therefore measurement probabilities.
- **Measurement:** returns one classical outcome; repeated shots estimate the distribution.
- In Quill, a Knot is still an 8-qubit register encoding a character. Superposition means the register is in a superposition of encoded basis states—not that the gate understands the character’s meaning.

Next: entangling qubits across Knots and exploring correlated outcomes.